# Explorador de sensibilidad (RED-100 Xe vs Ar)

Tres paneles interactivos que reutilizan, tal cual, las fórmulas ya validadas de
`fig_sensibilidad_mapa.py`, `tabla_fondo_Ar.py` y `nsi_core.py` (no se redefine
ninguna física nueva, solo se exponen los parámetros a un slider). Pensado para
ver en segundos qué pasa con $A_{90}$ o con el plano NSI al mover un parámetro.

Controles de referencia y la combinación exacta de widgets que los reproduce:
panel (a), modo **Ar + fondo declarado**, blanco Ar, $k{=}1$, $E{=}62$ kg·día:
fondo $\times0$ &rarr; $1{,}039$; fondo $\times1$ conocida &rarr; $1{,}38$.
Mismo panel, modo **ideal**, $k{=}4$, $E{=}192$ kg·día: blanco Xe &rarr; $1{,}48$;
blanco Ar &rarr; $1{,}03$ (cociente de tasas Ar/Xe $\times255$). Panel (c), que sí
arranca en sus valores por defecto ($t_{ON}{=}192$, $t_{OFF}{=}61$) &rarr; $75{,}6$;
subiendo $t_{ON}$ (p.ej. $10^9$) con $t_{OFF}{=}61$ fijo, la asíntota &rarr; $66$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, FloatLogSlider

BASE = "/home/oem/Desktop/Unipamplona/Trabajo de grado/Códigos/datos"
DCHI2_90 = 2.706   # 90% C.L., 1 g.d.l. (convención usada en toda la tesis)

## (a) $A_{90}$ frente a la exposición

Dos modos, cada uno copiado de su script: **ideal** usa la tasa acumulada
$R(N_e\geq k)$ de `sensib_ideal_{Xe,Ar}.dat` (fórmula de `fig_sensibilidad_mapa.py`,
$A_{90}=1+\sqrt{2{,}706/(R\,E)}$); **Ar + fondo declarado** usa los 5 bines de
`bkg_bins_Ar.dat` con la normalización gaussiana perfilada de `tabla_fondo_Ar.py`
($S_2=\sum S_k^2/(S_k{+}B_k)$, libre resta $P^2/Q$).

In [ ]:
def R_ideal(tag, k):                                              # fig_sensibilidad_mapa.py
    d = np.loadtxt(f"{BASE}/sensib_ideal_{tag}.dat", comments="#", dtype=str)
    return float(d[(d[:, 0] == str(k)) & (d[:, 1] == "Fnest")][0][2])

Ne_bk, Sk_bk, Bk0_bk = np.loadtxt(f"{BASE}/bkg_bins_Ar.dat", comments="#", unpack=True)  # tabla_fondo_Ar.py

@interact(modo=["ideal (sensib_ideal)", "Ar + fondo declarado (bkg_bins_Ar)"],
          blanco=["Xe", "Ar"], k=IntSlider(1, 1, 5, 1),
          E=FloatLogSlider(value=192, base=10, min=1, max=5, step=0.02, description="E [kg·día]"),
          mult=FloatSlider(0.0, min=0, max=10, step=0.1, description="fondo ×decl."),
          norm=["conocida", "libre"])
def panel_a(modo, blanco, k, E, mult, norm):
    if modo.startswith("ideal"):
        S2 = R_ideal(blanco, k) * E
    else:
        sel = Ne_bk >= k
        S, B = E * Sk_bk[sel], E * Bk0_bk[sel] * mult
        sig2 = S + B
        S2 = np.sum(S**2 / sig2)
        if mult > 0 and norm == "libre":
            P, Q = np.sum(S * B / sig2), np.sum(B**2 / sig2)
            S2 -= P**2 / Q
    A90 = 1.0 + np.sqrt(DCHI2_90 / S2)
    print(f"A90({blanco}, N_e>={k}, E={E:.0f} kg·día, {modo}) = {A90:.4f}")

## (b) Plano NSI: $A_-\leq A_{\rm amp}\leq A_+$

Usa `nsi_core.A_amp_grid` (la misma función de todas las figuras de lugares
ciegos/bandas). $Q_W$ se arma con la fórmula estándar del modelo estándar
electrodébil; $\sigma_A$ es la semianchura estadística ($A_{90}-1$ del panel
(a)) que define la banda permitida.

In [ ]:
import sys
sys.path.insert(0, "/home/oem/Desktop/Unipamplona/Trabajo de grado/Códigos/python")
from nsi_core import A_amp_grid                                   # nsi_core.py
S2W = 0.23857

@interact(Z=FloatSlider(54, min=1, max=92, step=1), N=FloatSlider(77.4, min=1, max=140, step=0.1),
          sigma_A=FloatSlider(0.05, min=0.001, max=1.0, step=0.001, readout_format=".3f"),
          flavor=["dV", "uV"], leptones=["ee-emu", "ee-ee"])
def panel_b(Z, N, sigma_A, flavor, leptones):
    QW = -N / 2.0 + (1.0 - 4.0 * S2W) / 2.0 * Z                   # constants.f90 (todas las carpetas)
    lx, ly = ("ee", "emu") if leptones == "ee-emu" else ("ee", "ee")
    Am, Ap = 1.0 - sigma_A, 1.0 + sigma_A
    g = np.linspace(-1.0, 1.0, 400)
    EX, EY = np.meshgrid(g, g)
    Amap = A_amp_grid(EX, EY, Z, N, QW, flavor, flavor, lx, ly)
    plt.figure(figsize=(4, 4))
    plt.contourf(EX, EY, (Amap >= Am) & (Amap <= Ap), levels=[0.5, 1], colors=["C0"])
    plt.xlabel(fr"$\varepsilon_{{{lx}}}^{{{flavor}}}$"); plt.ylabel(fr"$\varepsilon_{{{ly}}}^{{{flavor}}}$")
    plt.title(f"Z={Z:g}, N={N:g}: Q_W={QW:.3f}, A en [{Am:.3f}, {Ap:.3f}]")
    plt.show()

## (c) Xenón real (residuo ON$-$OFF)

$R_{ON,k}$, $R_{OFF,k}$ se reconstruyen por bin de PE desde el residuo
publicado, igual que en `fig_sensibilidad_mapa.py` (y en
`sensibilidad_exposicion_completa.py`): se resuelve
$R_{ON}-R_{OFF}=\Delta_{ON-OFF}$, $R_{ON}/t_{ON}^0+R_{OFF}/t_{OFF}^0=\sigma_{stat}^2$
con las exposiciones fiduciarias de 2024 ($t_{ON}^0=192$, $t_{OFF}^0=61$ kg·día).

In [ ]:
banda = np.loadtxt(f"{BASE}/chi2_ON_OFF_banda.dat", comments="#")   # fig_sensibilidad_mapa.py
RSM, dN, sig = banda[:, 1], banda[:, 4], banda[:, 5]
T_ON0, T_OFF0 = 192.0, 61.0
R_OFF = np.maximum((sig**2 - dN / T_ON0) / (1.0 / T_ON0 + 1.0 / T_OFF0), 0.0)
R_ON = np.maximum(dN + R_OFF, 0.0)

@interact(t_on=FloatLogSlider(value=192, base=10, min=1, max=9, step=0.02, description="t_ON [kg·día]"),
          t_off=FloatLogSlider(value=61, base=10, min=0, max=4, step=0.02, description="t_OFF [kg·día]"))
def panel_c(t_on, t_off):
    sig2 = R_ON / t_on + R_OFF / max(t_off, 1e-9)
    S2 = np.sum(RSM**2 / sig2)
    A90 = 1.0 + np.sqrt(DCHI2_90 / S2)
    print(f"A90(Xe real; t_ON={t_on:.0f}, t_OFF={t_off:.0f} kg·día) = {A90:.4f}")